# Week 9 - LangGraph 실습

## Part 1. 최소 그래프 실행

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. 상태 정의 (TypedDict)
class State(TypedDict):
    name: str
    msg: str

# 2. 노드 함수 2개 작성
def hello(state):
    return {"msg": f"Hello, {state['name']}!"}

def shout(state):
    return {"msg": state["msg"].upper()}

# 3. 그래프 빌더로 연결
g = StateGraph(State)
g.add_node("hello", hello)
g.add_node("shout", shout)
g.add_edge(START, "hello")
g.add_edge("hello", "shout")
g.add_edge("shout", END)

# 4. compile() 후 invoke로 실행
app = g.compile()
print(app.invoke({"name": "world"}))
# {'name': 'world', 'msg': 'HELLO, WORLD!'}

{'name': 'world', 'msg': 'HELLO, WORLD!'}


## Part 2. LangGraph 도구 등록

- LangGraph에서는 `@tool` 데코레이터로 함수를 도구로 표시한다.
- 도구 설명(docstring)은 LLM이 도구를 고를 때 보는 유일한 단서다 — 명확하고 구체적으로 적는다.

In [2]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """두 정수를 더해 결과를 돌려준다."""
    return a + b

@tool
def search_web(query: str) -> str:
    """주어진 쿼리로 웹 검색을 수행한다.

    실시간 정보(날씨, 뉴스, 환율 등)가 필요할 때 사용한다.
    """
    # 실습용 더미 응답
    return f"'{query}' 검색 결과: [더미 데이터] 관련 정보를 찾았습니다."

tools = [add, search_web]
# tools 리스트만 LLM에 넘기면 끝

print(add.name, "-", add.description)
print(search_web.name, "-", search_web.description)

add - 두 정수를 더해 결과를 돌려준다.
search_web - 주어진 쿼리로 웹 검색을 수행한다.

    실시간 정보(날씨, 뉴스, 환율 등)가 필요할 때 사용한다.


### 도구 연결 에이전트 그래프 구성

1. LLM에 도구를 `bind_tools()`로 연결한다.
2. 상태에 `messages` 리스트를 두고, 노드는 메시지를 추가하는 방식으로 동작한다.
3. LLM이 도구 호출을 요청하면 `ToolNode`가 실행하고, 결과를 다시 LLM에 전달한다.
4. LLM이 최종 답변을 생성하면 그래프가 종료된다.

In [3]:
import os
from typing import Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# 상태: messages 필드에 add_messages 리듀서 사용
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# LLM에 도구 연결
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm_with_tools = llm.bind_tools(tools)

# LLM 호출 노드
def call_llm(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# 그래프 구성
agent = StateGraph(AgentState)
agent.add_node("llm", call_llm)
agent.add_node("tools", ToolNode(tools))

agent.add_edge(START, "llm")
# tools_condition: tool_calls 있으면 "tools", 없으면 END
agent.add_conditional_edges("llm", tools_condition)
agent.add_edge("tools", "llm")   # 도구 실행 후 LLM으로 복귀

app = agent.compile()
print("에이전트 그래프 컴파일 완료")

d:\GitHub\1_lecture-materials\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


에이전트 그래프 컴파일 완료


In [4]:
from langchain_core.messages import HumanMessage

# 도구 호출이 필요한 질문
result = app.invoke({"messages": [HumanMessage(content="3 더하기 7은 얼마야?")]})

for msg in result["messages"]:
    print(f"[{msg.__class__.__name__}] {msg.content}")

[HumanMessage] 3 더하기 7은 얼마야?
[AIMessage] 
[ToolMessage] 10
[AIMessage] 3 더하기 7은 10입니다.


In [5]:
import os
from typing import Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# os.environ["GOOGLE_API_KEY"] = "AIza..."  # Google AI Studio에서 발급

# 상태: messages 필드에 add_messages 리듀서 사용
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# LLM에 도구 연결
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm_with_tools = llm.bind_tools(tools)

# LLM 호출 노드
def call_llm(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# 그래프 구성
agent = StateGraph(AgentState)
agent.add_node("llm", call_llm)
agent.add_node("tools", ToolNode(tools))

agent.add_edge(START, "llm")
# tools_condition: tool_calls 있으면 "tools", 없으면 END
agent.add_conditional_edges("llm", tools_condition)
agent.add_edge("tools", "llm")   # 도구 실행 후 LLM으로 복귀

app = agent.compile()
print("에이전트 그래프 컴파일 완료")

에이전트 그래프 컴파일 완료


### 도구 없이 질문하면? (실패 사례)

도구를 연결하지 않은 LLM에 동일한 질문을 던지면 어떻게 될까?

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

llm_plain = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

questions = [
    "3 더하기 7은 얼마야?",
    "오늘 서울 날씨 알려줘",
]

for q in questions:
    response = llm_plain.invoke([HumanMessage(content=q)])
    print(f"Q: {q}")
    print(f"A: {response.content}")
    print(f"tool_calls: {response.tool_calls}")  # 도구 호출 없음
    print()

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 48.526504141s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

---

## Part 3. 도구 선택 에이전트

도구가 여러 개일 때 LLM은 docstring을 보고 **어떤 도구를 쓸지 스스로 선택**한다.

- `add` — 덧셈
- `multiply` — 곱셈
- `get_current_time` — 현재 시각
- `search_web` — 실시간 웹 검색

같은 질문 패턴이라도 내용에 따라 다른 도구가 호출되는 것을 확인한다.

In [7]:
from datetime import datetime
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """두 정수를 더해 결과를 돌려준다."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱해 결과를 돌려준다."""
    return a * b

@tool
def get_current_time() -> str:
    """현재 날짜와 시각을 반환한다."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def search_web(query: str) -> str:
    """주어진 쿼리로 웹 검색을 수행한다.

    실시간 정보(날씨, 뉴스, 환율 등)가 필요할 때 사용한다.
    """
    return f"'{query}' 검색 결과: [더미 데이터] 관련 정보를 찾았습니다."

multi_tools = [add, multiply, get_current_time, search_web]
print("등록된 도구:", [t.name for t in multi_tools])

등록된 도구: ['add', 'multiply', 'get_current_time', 'search_web']


In [8]:
from typing import Annotated, TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

class MultiToolState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

llm_multi = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm_with_multi_tools = llm_multi.bind_tools(multi_tools)

def call_llm(state: MultiToolState):
    response = llm_with_multi_tools.invoke(state["messages"])
    return {"messages": [response]}

selector = StateGraph(MultiToolState)
selector.add_node("llm", call_llm)
selector.add_node("tools", ToolNode(multi_tools))
selector.add_edge(START, "llm")
selector.add_conditional_edges("llm", tools_condition)
selector.add_edge("tools", "llm")

selector_app = selector.compile()
print("도구 선택 에이전트 컴파일 완료")

도구 선택 에이전트 컴파일 완료


In [9]:
import time
from langchain_core.messages import HumanMessage, AIMessage

questions = [
    "3 더하기 7은 얼마야?",
    "5 곱하기 6은?",
    "지금 몇 시야?",
    "오늘 서울 날씨 알려줘",
]

# gemini-2.5-flash 무료 티어: 분당 5회 제한
# 에이전트는 질문 1개당 LLM을 2회 이상 호출하므로 질문 사이에 대기
SLEEP_SEC = 15

for i, q in enumerate(questions):
    result = selector_app.invoke({"messages": [HumanMessage(content=q)]})

    selected_tools = []
    for msg in result["messages"]:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                selected_tools.append(f"{tc['name']}({tc['args']})")

    final_answer = result["messages"][-1].content
    print(f"Q : {q}")
    print(f"선택된 도구: {' → '.join(selected_tools) if selected_tools else '없음'}")
    print(f"A : {final_answer}")
    print()

    if i < len(questions) - 1:
        print(f"  ({SLEEP_SEC}초 대기 중...)\n")
        time.sleep(SLEEP_SEC)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 29.653724569s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT